# Offline Document Q&A (RAG)

Notebook front-end for the `rag` package in this repo. Point `DOCS` at a PDF/TXT file or a folder of them, pick a retrieval mode and ask questions. Answers come with the file and page they were drawn from.

In [ ]:
# Uncomment on a fresh environment (e.g. Colab)
# !pip install -r requirements.txt

In [ ]:
from rag.chunking import chunk_pages
from rag.embeddings import SentenceTransformerEmbedder
from rag.generator import FlanT5Generator
from rag.loader import load_documents
from rag.pipeline import RAGPipeline

In [ ]:
DOCS = "examples/docs"   # a .pdf/.txt file or a folder
MODE = "hybrid"          # "dense", "bm25" or "hybrid"

pages = load_documents(DOCS)
chunks = chunk_pages(pages, chunk_size=200, overlap=40)
print(f"{len(pages)} pages -> {len(chunks)} chunks")

In [ ]:
embedder = None if MODE == "bm25" else SentenceTransformerEmbedder("all-MiniLM-L6-v2")
rag = RAGPipeline(chunks, embedder, FlanT5Generator("google/flan-t5-base"), mode=MODE, top_k=3)

In [ ]:
def ask(question):
    answer = rag.ask(question)
    print("Answer:", answer.text)
    for chunk, score in answer.sources:
        print(f"  - {chunk.source} (page {chunk.page}), score {score:.3f}")

ask("I just joined as an intern. Can I work from home?")